In [1]:
import piplite

In [2]:
await piplite.install('pandas')

In [5]:
import pandas as pd
import duckdb

In [6]:
con=duckdb.connect()

In [8]:
healthcare_patient_visits=pd.read_csv("healthcare_patient_analytics.csv")

In [9]:
queries = {
    "Total visits": """
        SELECT COUNT(*) AS total_visits
        FROM healthcare_patient_visits

    """,

    "Unique patients": """
        SELECT COUNT(DISTINCT patient_id) AS unique_patients
        FROM healthcare_patient_visits

    """,
    "Visits by department": """
        SELECT
            department,
            COUNT(*) AS total_visits
        FROM healthcare_patient_visits
        GROUP BY department
        ORDER BY total_visits DESC

    """,
    "Average cost by department": """
        SELECT
            department,
            ROUND(AVG(treatment_cost), 2) AS avg_cost
        FROM healthcare_patient_visits
        GROUP BY department
        ORDER BY avg_cost DESC;

    """,
    "Average length of stay": """
        SELECT
            department,
            ROUND(AVG(length_of_stay_days), 2) AS avg_los
        FROM healthcare_patient_visits
        GROUP BY department
        ORDER BY avg_los DESC

    """,
    "Find duplicate records": """
        SELECT
            patient_id,
            visit_date,
            department,
            treatment_type,
            COUNT(*) AS record_count
        FROM healthcare_patient_visits
        GROUP BY
            patient_id,
            visit_date,
            department,
            treatment_type
        HAVING COUNT(*) > 1

    """

}

results = {}

for heading, query in queries.items():
    results[heading] = con.execute(query).df()

for heading, df in results.items():
    print(f"\n{'=' * 50}")
    print(heading)
    print('=' * 50)
    print(df)


Total visits
   total_visits
0          5003

Unique patients
   unique_patients
0             5000

Visits by department
         department  total_visits
0       Orthopedics          1059
1        Cardiology           996
2  General Medicine           991
3        Pediatrics           989
4         Neurology           968

Average cost by department
         department  avg_cost
0         Neurology  55770.23
1        Cardiology  55632.09
2       Orthopedics  54915.95
3        Pediatrics  54781.58
4  General Medicine  53506.40

Average length of stay
         department  avg_los
0        Pediatrics     4.14
1  General Medicine     4.11
2         Neurology     4.06
3        Cardiology     4.03
4       Orthopedics     4.01

Find duplicate records
   patient_id        visit_date   department treatment_type  record_count
0        4886  23-07-2022 13:00   Cardiology    Observation             2
1        4995  28-07-2022 02:00  Orthopedics        Surgery             2
2        4997  28-07-